# 03. Experimentación y Selección de Modelos

Con los datos limpios, enriquecidos y escalados, es hora de encontrar el algoritmo predictivo ganador.

### Instrucciones Generales:
1. **Validación:** No entrenes y midas sobre el mismo conjunto (sobreajuste). Recuerda haber dividido en Entrenamiento y Prueba antes.
2. **Entrenamiento Base:** Entrena los siguientes modelos base con tu set de Entrenamiento y compáralos usando RMSE (Error Cuadrático Medio):
   - `LinearRegression`
   - `SGDRegressor`
   - `DecisionTreeRegressor`
   - `RandomForestRegressor`
3. **Cross Validation (Validación Cruzada):** Para tener una métrica robusta, usa `cross_val_score` en el set de Entrenamiento para cada uno de los modelos anteriores.
4. **Ajuste Fino (Fine Tuning):** Toma el modelo ganador y busca sus mejores hiperparámetros. Utiliza un `GridSearchCV` explorando el número de estimadores (`n_estimators`), las características máximas (`max_features`), etc.
5. **Conclusión y Benchmark (IMPORTANTE):** Redacta una conclusión comparando los algoritmos. Explica por qué escogiste el modelo final y valida tu decisión calculando el RMSE sobre tu Set de Prueba que habías reservado. Documenta si alguno de tus modelos se sobreajusto o subajusto. Recuerda que el modelo final no puede tener esos problemas!


In [270]:
# Empieza importando los algoritmos desde Scikit-Learn
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
import pandas as pd
import numpy as np


In [271]:
# Cargamos los datos procesados
df_procesado = pd.read_csv("../data/processed/train_processed.csv")

print("X_train:", df_procesado.shape)


X_train: (13209, 13)


1. Escalado y separación de target

In [272]:
# Escalado de variables
# dividir el conjunto de datos en características (X) y variable objetivo (y)
X_train = df_procesado.drop("median_house_value", axis=1)
y_train = df_procesado["median_house_value"]

# Escalar las variables numéricas para mejorar el rendimiento de los modelos de machine learning
scaler = StandardScaler()
numerical_features = X_train.select_dtypes(include="number").columns
X_train[numerical_features] = scaler.fit_transform(X_train[numerical_features])

2. Entrenamiento

In [280]:
# Definimos los 4 modelos base
modelos = {
    "LinearRegression":       LinearRegression(),
    "SGDRegressor":           SGDRegressor(random_state=7),
    "DecisionTreeRegressor":  DecisionTreeRegressor(random_state=42),
    "RandomForestRegressor":  RandomForestRegressor(random_state=6)
}

# Entrenamos cada modelo y calculamos RMSE sobre el train
print("Cálculo de RMSE sobre modelos de entrenamiento:")
print("-"*47)

rmse_modelos = {}

for nombre, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    prediccion = modelo.predict(X_train)
    error_train = mean_squared_error(y_train, prediccion)
    rmse = np.sqrt(error_train)
    rmse_modelos[nombre] = rmse
    print(f"{nombre:30s}: {rmse:,.0f}")

Cálculo de RMSE sobre modelos de entrenamiento:
-----------------------------------------------
LinearRegression              : 67,626
SGDRegressor                  : 67,729
DecisionTreeRegressor         : 0
RandomForestRegressor         : 18,623


3. Cross Validation

In [274]:
# Cross Validation con 5 folds para cada modelo
print("RMSE con Cross Validation (5 folds):")
print("-"*36)

for nombre, modelo in modelos.items():
    scores = cross_val_score(
        modelo, X_train, y_train,
        scoring="neg_mean_squared_error",
        cv=5
    )
    rmse_cv = np.sqrt(-scores).mean()
    std_cv  = np.sqrt(-scores).std()
    print(f"{nombre:30s}: {rmse_cv:,.0f} ± {std_cv:,.0f}")

RMSE con Cross Validation (5 folds):
------------------------------------
LinearRegression              : 67,780 ± 2,494
SGDRegressor                  : 14,717,797 ± 28,271,725
DecisionTreeRegressor         : 70,888 ± 2,931
RandomForestRegressor         : 50,664 ± 2,746


El mejor modelo en mi caso el Random Forest Regressor, ya que en la predicción por RMSE y Cross Validation obtiene el menor valor.

4. Fine Tuning - Ajuste fino

In [276]:
# Parámetros para grid search en RandomForestRegressor
parametros = {
    "n_estimators": [100, 150, 200],
    "min_samples_split": [2, 5, 6]
}

In [253]:
# grid search para RandomForestRegressor
grid_search = GridSearchCV(
    estimator=RandomForestRegressor(),
    param_grid=parametros,
    cv=5,
    scoring="neg_mean_squared_error"
)
grid_search.fit(X_train, y_train)
print("Mejores hiperparámetros:", grid_search.best_params_)
print(f"Mejor RMSE CV: ${np.sqrt(-grid_search.best_score_):,.0f}")

Mejores hiperparámetros: {'min_samples_split': 2, 'n_estimators': 200}
Mejor RMSE CV: $50,683


In [277]:
# Mejores hiperparámetros
mejores_hiperparametros = grid_search.best_params_
print("Mejores hiperparámetros:", mejores_hiperparametros)

Mejores hiperparámetros: {'min_samples_split': 2, 'n_estimators': 200}


### Benchmark y Conclusión Final
*(Escribe tus conclusiones de negocio aquí)*

In [278]:
import sys
sys.path.append("../src/features")
from build_features import preprocess_pipeline

val_df = pd.read_csv("../data/interim/val_set.csv")

# Preprocesamos el dataset de validación
val_procesado = preprocess_pipeline(val_df)

# dividir el conjunto de datos en características (X) y variable objetivo (y)
X_val_proc = val_procesado.drop(columns=["median_house_value"])
y_val_proc = val_procesado["median_house_value"]

# Escalar las variables numéricas del conjunto de validación utilizando el mismo scaler que se ajustó al conjunto de entrenamiento
X_val_proc[numerical_features] = scaler.transform(X_val_proc[numerical_features])

In [279]:
# Evaluamos el mejor modelo en validación
mejor_modelo = grid_search.best_estimator_
y_val_pred = mejor_modelo.predict(X_val_proc)
rmse_val = np.sqrt(mean_squared_error(y_val_proc, y_val_pred))

print(f"RMSE en validación: ${rmse_val:,.0f}")
print(f"RMSE en CV train:   ${np.sqrt(-grid_search.best_score_):,.0f}")

RMSE en validación: $51,854
RMSE en CV train:   $50,683


In [283]:
# Evaluación de los 4 modelos en validación
print("RMSE en conjunto de validación:")
print("-"*45)

rmse_val_modelos = {}

for nombre, modelo in modelos.items():
    pred_val = modelo.predict(X_val_proc)
    rmse_v = np.sqrt(mean_squared_error(y_val_proc, pred_val))
    rmse_val_modelos[nombre] = rmse_v
    print(f"{nombre:30s}: ${rmse_v:,.0f}")

RMSE en conjunto de validación:
---------------------------------------------
LinearRegression              : $69,124
SGDRegressor                  : $69,189
DecisionTreeRegressor         : $71,230
RandomForestRegressor         : $51,826


In [287]:
# DataFrame resumen
resumen = pd.DataFrame({
    "Model": list(rmse_val_modelos.keys()),
    "Train RMSE": [f"${rmse_modelos[k]:,.0f}" for k in rmse_val_modelos.keys()],
    "Validation RMSE": [f"${v:,.0f}" for v in rmse_val_modelos.values()]
})
resumen

,Model,Train RMSE,Validation RMSE
0,LinearRegression,"$67,626","$69,124"
1,SGDRegressor,"$67,729","$69,189"
2,DecisionTreeRegressor,$0,"$71,230"
3,RandomForestRegressor,"$18,623","$51,826"


In [290]:
# Evaluación con test_set
test_set_df = pd.read_csv("../data/interim/test_set.csv")

# Preprocesamos el dataset de test
test_procesado = preprocess_pipeline(test_set_df)

# dividir el conjunto de datos en características (X) y variable objetivo (y)
X_test_proc = test_procesado.drop(columns=["median_house_value"])
y_test_proc = test_procesado["median_house_value"]

# Escalar las variables numéricas del conjunto de test utilizando el mismo scaler que se ajustó al conjunto de entrenamiento
X_test_proc[numerical_features] = scaler.transform(X_test_proc[numerical_features])

In [292]:
# Evaluamos el mejor modelo en test
mejor_modelo = grid_search.best_estimator_
y_test_pred = mejor_modelo.predict(X_test_proc)
rmse_test = np.sqrt(mean_squared_error(y_test_proc, y_test_pred))

print(f"RMSE en test: ${rmse_test:,.0f}")
print(f"RMSE en validación: ${rmse_val:,.0f}")
print(f"RMSE en CV train:   ${np.sqrt(-grid_search.best_score_):,.0f}")

RMSE en test: $51,265
RMSE en validación: $51,854
RMSE en CV train:   $50,683


### Conclusiones:

- Para el dataset de Train, una Regresión Lineal (LinearRegression) arroja un error aproximado de $67,626.
- Modelos Avanzados: Al implementar diferentes modelos como: SGDRegressor, DecisionTreeRegressor y RandomForestRegressor, se obtuvieron diferentes errores.
  - SGDRegressor	$67,729
  - DecisionTreeRegressor $0
  - RandomForestRegressor $18,623

  Estos valores tenían variaciones al cambiar los hiperparámetro y random_state, en este caso se usaron los valores de hiperparámetros por defecto y random_state ajustado a cada modelo.
- En cross validation, se obtuvieron valores diferentes, mostrando los siguientes resultados:

  - LinearRegression              : 67,780 ± 2,494
  - SGDRegressor                  : 14,717,797 ± 28,271,725
  - DecisionTreeRegressor         : 70,888 ± 2,931
  - RandomForestRegressor         : 50,664 ± 2,746

- Considerando los modelos anteriores, se realizó la validación con el dataset val_set, obteniendo los siguientes resultados:

    Model	                Train RMSE	  Validation RMSE
  0	LinearRegression	    $67,626	      $69,124
  1	SGDRegressor	        $67,729	      $69,189
  2	DecisionTreeRegressor	$0	          $71,230
  3	RandomForestRegressor	$18,623	      $51,826

   ### Análisis de sesgo-varianza
    
  los datos, no generaliza → sobreajuste severo
    - **LinearRegression:** RMSE train y CV similares pero altos → modelo muy simple para este problema → subajuste
    - **SGDRegressor:** RMSE CV extremadamente alto → el cross validation nos muestra valores sumamente altos que muestra inestibilidad
    - **DecisionTree:** RMSE train $0 pero CV $70,888 → memorizó los datos
    - **RandomForest:** RMSE train y validación cercanos y más bajos que los otros modelos
  
  Con estos resultados podemos concluir que el modelo que mejores resultados presenta es Random Forest, por lo que es el modelo que se aplicará al dataset de pruebas final.

- El ajuste fino permitió obtener los hiperparámetros con mejores resultados para Random Forest y son:
  - 'min_samples_split': 2, 
  - 'n_estimators': 200

- En la evaluación final del test_set, se obtuvo un valor de $51,265, por lo que el modelo generaliza bien a datos completamente nuevos que nunca vio durante el entrenamiento.